# 🎨 TCS34725 센서 데이터 기반 MLP 색깔 분류

**목표**: 실제 RGB 센서(TCS34725)로 측정한 데이터를 사용하여 색깔 분류 MLP 모델 학습

**데이터**: 색깔 6종 × 50회 측정 = 300개 샘플
- 🔴 Red (빨강): 50개
- 🟠 Orange (주황): 50개
- 🟡 Yellow (노랑): 50개
- 🟢 Green (초록): 50개
- 🔵 Blue (파랑): 50개
- 🟣 Purple (보라): 50개

**모델**: PyTorch MLP

In [1]:
# 필요한 라이브러리 import
import numpy as np
import pandas as pd
import serial
import serial.tools.list_ports
import time
import os
from datetime import datetime

In [2]:
# 🔌 사용 가능한 시리얼 포트 확인
print("사용 가능한 시리얼 포트:")
for port in serial.tools.list_ports.comports():
    print(f"  • {port.device}: {port.description}")

사용 가능한 시리얼 포트:
  • COM3: USB-SERIAL CH340(COM3)


In [ ]:

SERIAL_PORT = "COM3"  
BAUD_RATE = 9600


SAMPLES_PER_COLOR = 50  # 색깔당 샘플 수
COLOR_NAMES = ['Red', 'Orange', 'Yellow', 'Green', 'Blue', 'Purple']
COLOR_KOREAN = ['빨강', '주황', '노랑', '초록', '파랑', '보라']
COLOR_EMOJI = ['🔴', '🟠', '🟡', '🟢', '🔵', '🟣']

print(f"수집 대상: {len(COLOR_NAMES)}가지 색깔")
print(f"색깔당 샘플: {SAMPLES_PER_COLOR}개")
print(f"총 샘플 수: {len(COLOR_NAMES) * SAMPLES_PER_COLOR}개")
print()
for i, (name, kr, emoji) in enumerate(zip(COLOR_NAMES, COLOR_KOREAN, COLOR_EMOJI)):
    print(f"  {i+1}. {emoji} {name} ({kr})")

수집 대상: 6가지 색깔
색깔당 샘플: 50개
총 샘플 수: 300개

  1. 🔴 Red (빨강)
  2. 🟠 Orange (주황)
  3. 🟡 Yellow (노랑)
  4. 🟢 Green (초록)
  5. 🔵 Blue (파랑)
  6. 🟣 Purple (보라)


In [ ]:
# 데이터 저장 리스트
collected_data = []
import re

def parse_rgb_data(line):
    """
    다양한 형식의 RGB 데이터를 파싱
    
    지원 형식:
    - "R : 3 G : 2 B : 1"
    - "R: 3 G: 2 B: 1"
    - "145,58,52"
    """
    # 형식 1: "R : 값 G : 값 B : 값" 또는 "R: 값 G: 값 B: 값"
    pattern = r'R\s*:\s*(\d+)\s*G\s*:\s*(\d+)\s*B\s*:\s*(\d+)'
    match = re.search(pattern, line, re.IGNORECASE)
    if match:
        r, g, b = int(match.group(1)), int(match.group(2)), int(match.group(3))
        return r, g, b
    
    # 형식 2: "값,값,값"
    if ',' in line:
        parts = line.split(',')
        if len(parts) >= 3:
            try:
                r, g, b = float(parts[0]), float(parts[1]), float(parts[2])
                return int(r), int(g), int(b)
            except:
                pass
    
    return None

def collect_color_data(color_idx):
    """
    특정 색깔의 RGB 데이터를 수집하는 함수
    
    Args:
        color_idx: 색깔 인덱스 (0~5)
    
    Returns:
        list: 수집된 RGB 데이터 리스트
    """
    color_name = COLOR_NAMES[color_idx]
    color_kr = COLOR_KOREAN[color_idx]
    color_emoji = COLOR_EMOJI[color_idx]
    
    data_list = []
    
    try:
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=2)
        print(f"\n{'='*60}")
        print(f"{color_emoji} {color_name} ({color_kr}) 데이터 수집")
        print(f"{'='*60}")
        print(f"✓ 연결됨: {SERIAL_PORT} @ {BAUD_RATE} baud")
        print(f"\n 목표: {SAMPLES_PER_COLOR}개 샘플 수집")
        print(f" [엔터] 측정 | [q + 엔터] 종료")
        print(f"{'-'*60}")
        
        time.sleep(2)  # 아두이노 안정화 대기
        ser.reset_input_buffer()
        count = 0
        
        while count < SAMPLES_PER_COLOR:
            remaining = SAMPLES_PER_COLOR - count
            user_input = input(f"\n[{count+1}/{SAMPLES_PER_COLOR}] 엔터 = 측정, q = 종료 > ").strip().lower()
            
            if user_input == 'q':
                print("\n 수집 중단")
                break
            
            # 버퍼 비우기
            ser.reset_input_buffer()
            time.sleep(0.1)
            
            print(" 측정 중...", end=" ", flush=True)
            
            # 여러 번 시도
            measured = False
            for attempt in range(10):  # 최대 10번 시도
                if ser.in_waiting > 0:
                    try:
                        line = ser.readline().decode('utf-8', errors='ignore').strip()
                        
                        if line:
                            result = parse_rgb_data(line)
                            if result:
                                r, g, b = result
                                if 0 <= r <= 255 and 0 <= g <= 255 and 0 <= b <= 255:
                                    count += 1
                                    data_list.append({
                                        'R': r,
                                        'G': g,
                                        'B': b,
                                        'label': color_name
                                    })
                                    print(f"\n RGB({r:3d}, {g:3d}, {b:3d}) - 샘플 #{count}")
                                    measured = True
                                    break
                    except Exception as e:
                        print(f"[오류: {e}]", end=" ")
                
                time.sleep(0.2)  # 200ms 대기
            
            if not measured:
                print("\n 데이터 수신 실패. 다시 시도하세요.")
        
        print(f"\n{'-'*60}")
        print(f" {color_emoji} {color_name} 수집 완료: {count}개")
        
    except serial.SerialException as e:
        print(f" 연결 실패: {e}")
    finally:
        if 'ser' in locals() and ser.is_open:
            ser.close()
            print(" 시리얼 포트 닫힘")
    
    return data_list

---
## 📥 데이터 수집 실행

아래 셀을 실행하여 데이터를 수집합니다.  
각 색깔별로 50개씩, 총 300개의 RGB 데이터를 수집합니다.

In [ ]:
# 🎨 색깔 선택 및 수집 실행
print("="*60)
print("🎨 수집할 색깔을 선택하세요")
print("="*60)
for i, (name, kr, emoji) in enumerate(zip(COLOR_NAMES, COLOR_KOREAN, COLOR_EMOJI)):
    print(f"  {i+1}. {emoji} {name} ({kr})")
print("  0.  취소")
print("="*60)

try:
    choice = int(input("\n색깔 번호 입력 (0=취소, 1~6): "))
    if choice == 0:
        print("\n취소되었습니다.")
    elif 1 <= choice <= 6:
        color_idx = choice - 1
        emoji = COLOR_EMOJI[color_idx]
        name = COLOR_NAMES[color_idx]
        kr = COLOR_KOREAN[color_idx]
        print(f"\n✓ 선택: {emoji} {name} ({kr})")
        print(f"✓ 50개 샘플을 수집합니다.")
        print(f"\n⚠️ 종료하려면 'q' 입력 후 엔터\n")
        
        # 데이터 수집
        collected_data = collect_color_data(color_idx)
        
        # 수집 완료 후 자동 저장
        if collected_data:
            save_dir = "color_data"
            os.makedirs(save_dir, exist_ok=True)
            
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{save_dir}/{name.lower()}_rgb_data_{timestamp}.csv"
            
            df_single = pd.DataFrame(collected_data)
            df_single.to_csv(filename, index=False, encoding='utf-8')
            
            print(f"\n{'='*60}")
            print(f" 데이터 저장 완료!")
            print(f" 파일: {filename}")
            print(f" 수집된 샘플: {len(df_single)}개")
            print(f"{'='*60}")
        else:
            print("\n 수집된 데이터가 없습니다.")
    else:
        print(" 0~6 사이의 숫자를 입력하세요.")
except ValueError:
    print(" 숫자를 입력하세요.")
except KeyboardInterrupt:
    print("\n\n 사용자가 중단했습니다. (Ctrl+C)")

🎨 수집할 색깔을 선택하세요
  1. 🔴 Red (빨강)
  2. 🟠 Orange (주황)
  3. 🟡 Yellow (노랑)
  4. 🟢 Green (초록)
  5. 🔵 Blue (파랑)
  6. 🟣 Purple (보라)
  0. ❌ 취소

✓ 선택: 🟣 Purple (보라)
✓ 50개 샘플을 수집합니다.

⚠️ 종료하려면 'q' 입력 후 엔터


🟣 Purple (보라) 데이터 수집
✓ 연결됨: COM3 @ 9600 baud

📌 목표: 50개 샘플 수집
📌 [엔터] 측정 | [q + 엔터] 종료
------------------------------------------------------------
⏳ 측정 중... 
✅ RGB(125,  62,  66) - 샘플 #1
⏳ 측정 중... 
✅ RGB(118,  68,  68) - 샘플 #2
⏳ 측정 중... 
✅ RGB(120,  67,  66) - 샘플 #3
⏳ 측정 중... 
✅ RGB(122,  67,  65) - 샘플 #4
⏳ 측정 중... 
✅ RGB(124,  65,  64) - 샘플 #5
⏳ 측정 중... 
✅ RGB(127,  64,  63) - 샘플 #6
⏳ 측정 중... 
✅ RGB(128,  63,  62) - 샘플 #7
⏳ 측정 중... 
✅ RGB(130,  63,  61) - 샘플 #8
⏳ 측정 중... 
✅ RGB(128,  63,  62) - 샘플 #9
⏳ 측정 중... 
✅ RGB(128,  63,  62) - 샘플 #10
⏳ 측정 중... 
✅ RGB(130,  63,  61) - 샘플 #11
⏳ 측정 중... 
✅ RGB(128,  64,  62) - 샘플 #12
⏳ 측정 중... 
✅ RGB(122,  66,  65) - 샘플 #13
⏳ 측정 중... 
✅ RGB(124,  65,  64) - 샘플 #14
⏳ 측정 중... 
✅ RGB(126,  64,  63) - 샘플 #15
⏳ 측정 중... 
✅ RGB(126,  64,  63) - 샘플 #16
⏳ 측정 중... 

In [ ]:
#  현재 수집된 데이터 확인
if collected_data:
    df_current = pd.DataFrame(collected_data)
    
    print("="*50)
    print(" 현재 수집된 데이터")
    print("="*50)
    print(f"샘플 수: {len(df_current)}개")
    print(f"색깔: {df_current['label'].iloc[0]}")
    print(f"\nRGB 통계:")
    print(df_current[['R', 'G', 'B']].describe().round(1))
    print(f"\n데이터 미리보기:")
    display(df_current)
else:
    print(" 수집된 데이터가 없습니다.")

📊 현재 수집된 데이터
샘플 수: 50개
색깔: Purple

RGB 통계:
           R     G     B
count   50.0  50.0  50.0
mean   128.4  63.0  62.0
std      4.6   2.6   2.3
min    117.0  50.0  55.0
25%    127.0  62.0  61.0
50%    129.0  63.0  62.0
75%    130.8  64.0  63.0
max    149.0  68.0  68.0

데이터 미리보기:


,R,G,B,label
0,125,62,66,Purple
1,118,68,68,Purple
2,120,67,66,Purple
3,122,67,65,Purple
4,124,65,64,Purple
5,127,64,63,Purple
6,128,63,62,Purple
7,130,63,61,Purple
8,128,63,62,Purple
9,128,63,62,Purple


---
## 📂 전체 데이터 병합

모든 색깔의 CSV 파일을 하나로 합칩니다.

In [ ]:
#  저장된 CSV 파일 목록 확인
save_dir = "color_data"
if os.path.exists(save_dir):
    files = [f for f in os.listdir(save_dir) if f.endswith('.csv')]
    if files:
        print("="*60)
        print(" 저장된 데이터 파일 목록")
        print("="*60)
        
        total_samples = 0
        color_counts = {name: 0 for name in COLOR_NAMES}
        
        for i, f in enumerate(sorted(files)):
            filepath = f"{save_dir}/{f}"
            temp_df = pd.read_csv(filepath)
            size = os.path.getsize(filepath) / 1024
            label = temp_df['label'].iloc[0] if len(temp_df) > 0 else "Unknown"
            emoji = COLOR_EMOJI[COLOR_NAMES.index(label)] if label in COLOR_NAMES else "❓"
            print(f"  {i+1}. {emoji} {f} ({len(temp_df)}개, {size:.1f}KB)")
            total_samples += len(temp_df)
            if label in color_counts:
                color_counts[label] += len(temp_df)
        
        print("="*60)
        print(f" 총 파일 수: {len(files)}개")
        print(f" 총 샘플 수: {total_samples}개")
        print("\n색깔별 수집 현황:")
        for name, kr, emoji in zip(COLOR_NAMES, COLOR_KOREAN, COLOR_EMOJI):
            count = color_counts[name]
            status = "✅" if count >= 50 else "⏳"
            print(f"  {emoji} {name} ({kr}): {count}/50개 {status}")
    else:
        print("저장된 CSV 파일이 없습니다.")
else:
    print(f"'{save_dir}' 폴더가 없습니다.")

📂 저장된 데이터 파일 목록
  1. 🔵 blue_rgb_data_20260121_215233.csv (50개, 0.8KB)
  2. 🟢 green_rgb_data_20260121_215048.csv (50개, 0.8KB)
  3. 🟠 orange_rgb_data_20260121_214637.csv (50개, 0.9KB)
  4. 🟣 purple_rgb_data_20260121_215401.csv (50개, 0.9KB)
  5. 🔴 red_rgb_data_20260121_214253.csv (50개, 0.7KB)
  6. 🟡 yellow_rgb_data_20260121_214838.csv (50개, 0.9KB)
📊 총 파일 수: 6개
📊 총 샘플 수: 300개

색깔별 수집 현황:
  🔴 Red (빨강): 50/50개 ✅
  🟠 Orange (주황): 50/50개 ✅
  🟡 Yellow (노랑): 50/50개 ✅
  🟢 Green (초록): 50/50개 ✅
  🔵 Blue (파랑): 50/50개 ✅
  🟣 Purple (보라): 50/50개 ✅


In [ ]:
# 🔗 모든 CSV 파일 병합
save_dir = "color_data"
if os.path.exists(save_dir):
    files = [f for f in os.listdir(save_dir) if f.endswith('.csv') and not f.startswith('all_')]
    
    if files:
        all_data = []
        for f in files:
            temp_df = pd.read_csv(f"{save_dir}/{f}")
            all_data.append(temp_df)
        
        df_all = pd.concat(all_data, ignore_index=True)
        
        print("="*60)
        print(" 병합된 전체 데이터")
        print("="*60)
        print(f"총 샘플 수: {len(df_all)}개")
        print(f"\n클래스별 분포:")
        print(df_all['label'].value_counts())
        
        # 병합 파일 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        merged_filename = f"{save_dir}/all_colors_rgb_data_{timestamp}.csv"
        df_all.to_csv(merged_filename, index=False, encoding='utf-8')
        
        print(f"\n 병합 파일 저장: {merged_filename}")
    else:
        print("병합할 CSV 파일이 없습니다.")
else:
    print(f"'{save_dir}' 폴더가 없습니다.")

📊 병합된 전체 데이터
총 샘플 수: 300개

클래스별 분포:
label
Blue      50
Green     50
Orange    50
Purple    50
Red       50
Yellow    50
Name: count, dtype: int64

💾 병합 파일 저장: color_data/all_colors_rgb_data_20260121_215831.csv
